# Decomposicao Classica de Series Temporais

Este notebook apresenta a **decomposicao classica** de series temporais usando a biblioteca `chronobox`.

A decomposicao classica separa uma serie temporal em tres componentes:
- **Tendencia (T)**: movimento de longo prazo
- **Sazonalidade (S)**: padroes que se repetem em intervalos fixos
- **Residuo (R)**: variacao aleatoria restante

Dois modelos sao considerados:
- **Aditivo**: $Y_t = T_t + S_t + R_t$
- **Multiplicativo**: $Y_t = T_t \times S_t \times R_t$

## Setup e Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox import ClassicalDecomposition
from chronobox.visualization import plot_decomposition

import sys
sys.path.insert(0, '..')
from utils.plot_helpers import plot_decomposition as plot_decomp_helper
from utils.plot_helpers import plot_seasonal_subseries
from utils.data_generators import generate_additive_seasonal, generate_multiplicative_seasonal

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Teoria: Media Movel para Estimacao de Tendencia

A decomposicao classica usa **medias moveis centradas** para estimar a tendencia.

Para uma serie com periodo sazonal $m$:
- Se $m$ e **impar**: media movel simples centrada de ordem $m$
- Se $m$ e **par**: media movel $2 \times m$ (com pesos $1/2$ nas extremidades)

$$\hat{T}_t = \frac{1}{2m}\left(\frac{1}{2}y_{t-m/2} + y_{t-m/2+1} + \cdots + y_{t+m/2-1} + \frac{1}{2}y_{t+m/2}\right)$$

Isso remove a componente sazonal e suaviza o ruido, deixando apenas a tendencia.

## 2. Dados Sinteticos: Modelo Aditivo

Vamos comecar gerando dados com estrutura aditiva conhecida para entender o algoritmo.

In [ ]:
# Gerar dados aditivos sinteticos: Y = T + S + R
y_add, trend_true, seasonal_true, resid_true = generate_additive_seasonal(
    n=120, trend_slope=0.5, seasonal_amp=10.0, noise_std=2.0, s=12, seed=42
)

dates = pd.date_range('2010-01', periods=120, freq='MS')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(dates, y_add, 'k-', linewidth=0.8)
axes[0, 0].set_title('Serie Observada (Y = T + S + R)')
axes[0, 1].plot(dates, trend_true, 'b-', linewidth=1.2)
axes[0, 1].set_title('Tendencia Verdadeira')
axes[1, 0].plot(dates, seasonal_true, 'g-', linewidth=0.8)
axes[1, 0].set_title('Sazonalidade Verdadeira')
axes[1, 1].plot(dates, resid_true, 'r-', linewidth=0.8)
axes[1, 1].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[1, 1].set_title('Residuo Verdadeiro')
fig.suptitle('Componentes Verdadeiros (Modelo Aditivo)', fontsize=14)
fig.tight_layout()
plt.show()

## 3. Decomposicao Aditiva com chronobox

No modelo aditivo, assumimos $Y_t = T_t + S_t + R_t$.

O algoritmo:
1. Estima $\hat{T}_t$ via media movel centrada
2. Calcula a serie dessazonalizada: $Y_t - \hat{T}_t$
3. Estima $\hat{S}_t$ como a media de cada "estacao" (mes, trimestre, etc.)
4. Normaliza $\hat{S}_t$ para que $\sum_{j=1}^{m} \hat{S}_j = 0$
5. Residuo: $\hat{R}_t = Y_t - \hat{T}_t - \hat{S}_t$

In [ ]:
# Decomposicao classica aditiva
cd_add = ClassicalDecomposition(period=12, model='additive')
result_add = cd_add.fit(y_add)

print(result_add.summary())
print(f"\nModelo: {result_add.model}")
print(f"Periodo: {result_add.period}")

In [ ]:
# Visualizacao dos componentes estimados
fig = plot_decomposition(result_add, title='Decomposicao Classica Aditiva', dates=dates)
plt.show()

### Interpretacao Visual

- **Observed**: a serie original com tendencia crescente e sazonalidade regular
- **Trend**: captura o movimento ascendente de longo prazo. Note os NaN nas extremidades (a media movel perde $m/2$ pontos em cada lado)
- **Seasonal**: padroes repetitivos com amplitude constante. No modelo aditivo, os fatores sazonais somam zero
- **Remainder**: ruido aleatorio sem estrutura aparente. Se houver padrao, o modelo pode ser inadequado

In [ ]:
# Verificacao: T + S + R = Y
reconstruction = result_add.trend + result_add.seasonal + result_add.remainder
valid = ~np.isnan(reconstruction)
max_error = np.max(np.abs(reconstruction[valid] - y_add[valid]))
print(f"Erro maximo de reconstrucao (aditivo): {max_error:.2e}")
print(f"Verificacao T + S + R = Y: {'OK' if max_error < 1e-10 else 'FALHOU'}")

## 4. Estimacao de Fatores Sazonais

Os fatores sazonais sao calculados pela media de cada mes (ou estacao) apos remover a tendencia. No modelo aditivo, os fatores sazonais sao normalizados para somar zero.

In [ ]:
# Fatores sazonais: um valor por mes
seasonal_factors = result_add.seasonal[:12]
month_names = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
               'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if v >= 0 else 'salmon' for v in seasonal_factors]
ax.bar(month_names, seasonal_factors, color=colors, edgecolor='gray')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Fatores Sazonais Estimados (Modelo Aditivo)')
ax.set_ylabel('Fator Sazonal')
print(f"Soma dos fatores sazonais: {seasonal_factors.sum():.6f} (deve ser ~0)")
plt.show()

### Interpretacao dos Fatores Sazonais

Cada barra representa o **desvio tipico** daquele mes em relacao a tendencia:
- Barras positivas: meses com valores **acima** da tendencia
- Barras negativas: meses com valores **abaixo** da tendencia
- A soma e zero, garantindo que a sazonalidade nao afeta o nivel medio

## 5. Dados Reais: Airline Passengers

O dataset classico de Box-Jenkins com passageiros de companhias aereas (1949-1960). Esta serie e um exemplo tipico de sazonalidade **multiplicativa**: a amplitude sazonal cresce com o nivel.

In [ ]:
# Carregar dados
airline = pd.read_csv('../data/airline.csv', parse_dates=['date'])
y_airline = airline['passengers'].values
dates_airline = pd.DatetimeIndex(airline['date'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dates_airline, y_airline, 'k-', linewidth=0.8)
ax.set_title('Airline Passengers (1949-1960)')
ax.set_ylabel('Numero de Passageiros (milhares)')
ax.set_xlabel('Ano')
plt.show()

print(f"Observacoes: {len(y_airline)}")
print(f"Periodo: {dates_airline[0].strftime('%Y-%m')} a {dates_airline[-1].strftime('%Y-%m')}")
print(f"Min: {y_airline.min()}, Max: {y_airline.max()}")

## 6. Decomposicao Multiplicativa

No modelo multiplicativo, assumimos $Y_t = T_t \times S_t \times R_t$.

O algoritmo:
1. Estima $\hat{T}_t$ via media movel centrada (igual ao aditivo)
2. Calcula a razao: $Y_t / \hat{T}_t$
3. Estima $\hat{S}_t$ como a media de cada estacao
4. Normaliza $\hat{S}_t$ para que $\frac{1}{m}\sum_{j=1}^{m} \hat{S}_j = 1$
5. Residuo: $\hat{R}_t = Y_t / (\hat{T}_t \times \hat{S}_t)$

**Quando usar?** Quando a amplitude sazonal cresce proporcionalmente ao nivel da serie.

In [ ]:
# Decomposicao multiplicativa do airline
cd_mult = ClassicalDecomposition(period=12, model='multiplicative')
result_mult = cd_mult.fit(y_airline)

print(result_mult.summary())

In [ ]:
# Visualizacao
fig = plot_decomposition(result_mult, title='Decomposicao Multiplicativa - Airline', dates=dates_airline)
plt.show()

### Interpretacao Visual (Multiplicativo)

- **Trend**: crescimento exponencial do numero de passageiros ao longo dos anos
- **Seasonal**: fatores centrados em 1.0. Valores > 1 indicam meses acima da tendencia (verao), < 1 abaixo (inverno)
- **Remainder**: residuos centrados em 1.0. Devem ser ruido sem padrao sistematico

Note como os fatores sazonais multiplicativos capturam a proporcionalidade: o pico de verao representa ~20% acima da tendencia, independente do nivel.

In [ ]:
# Verificacao: T * S * R = Y
recon_mult = result_mult.trend * result_mult.seasonal * result_mult.remainder
valid_mult = ~np.isnan(recon_mult)
max_error_mult = np.max(np.abs(recon_mult[valid_mult] - y_airline[valid_mult]))
print(f"Erro maximo de reconstrucao (multiplicativo): {max_error_mult:.2e}")

# Fatores sazonais multiplicativos
sf_mult = result_mult.seasonal[:12]
print(f"\nFatores sazonais multiplicativos:")
for m, f in zip(month_names, sf_mult):
    pct = (f - 1) * 100
    print(f"  {m}: {f:.4f} ({pct:+.1f}% em relacao a tendencia)")
print(f"\nMedia dos fatores: {sf_mult.mean():.6f} (deve ser ~1.0)")

## 7. Comparacao: Aditivo vs Multiplicativo no Airline

Aplicar ambos os modelos ao mesmo dataset para entender quando cada um e mais adequado.

In [ ]:
# Aditivo no airline (para comparacao)
cd_add_airline = ClassicalDecomposition(period=12, model='additive')
result_add_airline = cd_add_airline.fit(y_airline)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('Aditivo (esquerda) vs Multiplicativo (direita) - Airline', fontsize=14)

# Tendencia
axes[0, 0].plot(dates_airline, result_add_airline.trend, 'b-', linewidth=1)
axes[0, 0].set_title('Tendencia (Aditivo)')
axes[0, 1].plot(dates_airline, result_mult.trend, 'b-', linewidth=1)
axes[0, 1].set_title('Tendencia (Multiplicativo)')

# Sazonal
axes[1, 0].plot(dates_airline, result_add_airline.seasonal, 'g-', linewidth=0.8)
axes[1, 0].set_title('Sazonal (Aditivo) - amplitude constante')
axes[1, 1].plot(dates_airline, result_mult.seasonal, 'g-', linewidth=0.8)
axes[1, 1].set_title('Sazonal (Multiplicativo) - fatores proporcionais')

# Residuo
axes[2, 0].plot(dates_airline, result_add_airline.remainder, 'r-', linewidth=0.8)
axes[2, 0].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[2, 0].set_title('Residuo (Aditivo)')
axes[2, 1].plot(dates_airline, result_mult.remainder, 'r-', linewidth=0.8)
axes[2, 1].axhline(1, color='gray', linestyle='--', linewidth=0.5)
axes[2, 1].set_title('Residuo (Multiplicativo)')

fig.tight_layout()
plt.show()

### Interpretacao da Comparacao

**Observe os residuos**:
- No modelo **aditivo**, os residuos mostram um padrao claro: variancia crescente ao longo do tempo (heterocedasticidade). Isso indica que o modelo aditivo e **inadequado** para esta serie.
- No modelo **multiplicativo**, os residuos sao mais homogeneos, sem padrao claro de variancia.

**Regra pratica**: se a amplitude sazonal cresce com o nivel, use multiplicativo. Se a amplitude e constante, use aditivo.

In [ ]:
# Analise quantitativa dos residuos
valid_add = ~np.isnan(result_add_airline.remainder)
valid_m = ~np.isnan(result_mult.remainder)

resid_add = result_add_airline.remainder[valid_add]
resid_mult = result_mult.remainder[valid_m]

# Dividir em primeira e segunda metade para verificar homocedasticidade
mid = len(resid_add) // 2
print("Desvio-padrao dos residuos por metade da serie:")
print(f"  Aditivo:         1a metade = {resid_add[:mid].std():.2f}, 2a metade = {resid_add[mid:].std():.2f}")
print(f"  Multiplicativo:  1a metade = {resid_mult[:mid].std():.4f}, 2a metade = {resid_mult[mid:].std():.4f}")
print(f"\nRazao (2a/1a metade):")
print(f"  Aditivo:         {resid_add[mid:].std() / resid_add[:mid].std():.2f}x")
print(f"  Multiplicativo:  {resid_mult[mid:].std() / resid_mult[:mid].std():.2f}x")
print("\nValores proximos de 1.0 indicam homocedasticidade (melhor ajuste).")

## 8. Subseries Sazonais

Uma forma alternativa de visualizar os fatores sazonais e atraves de **subseries sazonais**, onde agrupamos os valores por mes.

In [ ]:
# Subseries sazonais para a decomposicao multiplicativa do airline
fig = plot_seasonal_subseries(
    result_mult.seasonal, s=12, labels=month_names,
    title='Subseries Sazonais - Airline (Multiplicativo)'
)
plt.show()

## 9. Limitacoes da Decomposicao Classica

A decomposicao classica tem limitacoes importantes:

1. **Perda de dados**: a media movel causa NaN nas $m/2$ primeiras e ultimas observacoes
2. **Sazonalidade fixa**: assume que os fatores sazonais sao identicos ano a ano
3. **Sensivel a outliers**: medias moveis simples nao sao robustas
4. **Apenas dois modelos**: aditivo ou multiplicativo, sem flexibilidade intermediaria

Essas limitacoes sao superadas por metodos mais avancados como STL e X-13 (proximos notebooks).

In [ ]:
# Demonstracao: NaN nas extremidades
n_nan_start = np.argmax(~np.isnan(result_mult.trend))
n_nan_end = len(result_mult.trend) - np.argmax(~np.isnan(result_mult.trend[::-1]))
n_nan = np.sum(np.isnan(result_mult.trend))

print(f"Total de observacoes: {len(result_mult.trend)}")
print(f"Observacoes com NaN na tendencia: {n_nan}")
print(f"Primeiras {n_nan_start} e ultimas {len(result_mult.trend) - n_nan_end} sao NaN")
print(f"Perda de dados: {n_nan/len(result_mult.trend)*100:.1f}% da serie")

## Exercicio

### Exercicio 1: Decomposicao Aditiva vs Multiplicativa do Airline

1. Aplique a decomposicao **aditiva** e **multiplicativa** ao dataset `airline.csv`
2. Compare os residuos de ambos os modelos
3. Calcule o desvio-padrao dos residuos para a 1a e 2a metade da serie
4. Qual modelo e mais adequado? Justifique com base na homocedasticidade dos residuos

### Exercicio 2: Dados Sinteticos Multiplicativos

1. Use `generate_multiplicative_seasonal()` para criar dados com sazonalidade multiplicativa
2. Aplique decomposicao aditiva e multiplicativa
3. Compare as tendencias estimadas com a tendencia verdadeira
4. Qual modelo recupera melhor a tendencia?

In [ ]:
# Exercicio 1 - Espaco para resolucao
# Dica: use ClassicalDecomposition(period=12, model='additive') e model='multiplicative'
# Compare os std dos residuos em cada metade da serie

# Seu codigo aqui:


In [ ]:
# Exercicio 2 - Espaco para resolucao
# Dica: use generate_multiplicative_seasonal(n=144, seed=42)
# Compare result.trend com a tendencia verdadeira

# Seu codigo aqui:
